# Rutas, distancias y tiempos de viaje (GTFS)

Este notebook calcula, para cada estudiante beneficiario, la **mejor conexión en transporte público** (SITP + TransMilenio) entre su domicilio y su Institución de Educación Superior (IES), usando un feed **GTFS**.

**Salida principal:** `outputs/estudiantes_con_rutas.xlsx`, con las variables de interés `dist_total_estimada_km` y `tiempo_total_min` (columnas `conn_*`) que alimentan el análisis descriptivo y espacial del proyecto.

**Insumos esperados en `data/raw/`:**
- `GTFS-2026-04-29/` — feed GTFS de SITP + TransMilenio (paraderos, rutas, viajes, horarios).
- `estudiantes.xlsx` — columnas `lat`, `lon` (domicilio) y `uni_lat`, `uni_lon` (IES) por estudiante.

**Orden de ejecución:** correr las celdas de este notebook de arriba hacia abajo. No requiere argumentos; las rutas se resuelven automáticamente de forma relativa al repositorio (ver celda de configuración).


In [1]:
import argparse
import zipfile
import io
import os
import sys
import pandas as pd
import numpy as np
from shapely.geometry import LineString, Point
from shapely.strtree import STRtree
from pyproj import Transformer
from scipy.spatial import cKDTree

CRS_METRICO = "EPSG:3116"
CRS_GEO = "EPSG:4326"


def leer_tabla_gtfs(z: zipfile.ZipFile, nombre: str) -> pd.DataFrame:
    candidatos = [n for n in z.namelist() if n.endswith(nombre)]
    if not candidatos:
        raise FileNotFoundError(f"No encontré '{nombre}' dentro del GTFS.")
    with z.open(candidatos[0]) as f:
        return pd.read_csv(
            io.TextIOWrapper(f, encoding="utf-8-sig"),
            dtype={"shape_id": str, "route_id": str, "trip_id": str, "stop_id": str},
        )


def leer_tabla_gtfs_carpeta(carpeta: str, nombre: str) -> pd.DataFrame:
    ruta_exacta = os.path.join(carpeta, nombre)
    if os.path.isfile(ruta_exacta):
        return pd.read_csv(ruta_exacta, encoding="utf-8-sig", dtype={"shape_id": str, "route_id": str, "trip_id": str, "stop_id": str})

    candidatos = [f for f in os.listdir(carpeta) if f.endswith(nombre)]
    if not candidatos:
        raise FileNotFoundError(f"No encontré '{nombre}' en la carpeta '{carpeta}'.")
    return pd.read_csv(os.path.join(carpeta, candidatos[0]), encoding="utf-8-sig", dtype={"shape_id": str, "route_id": str, "trip_id": str, "stop_id": str})


def leer_tabla_gtfs_carpeta_opcional(carpeta: str, nombre: str) -> pd.DataFrame:
    try:
        return leer_tabla_gtfs_carpeta(carpeta, nombre)
    except FileNotFoundError:
        return None


def cargar_gtfs(ruta: str, cargar_horarios: bool = False) -> dict:
    """
    Carga el GTFS. Si cargar_horarios=True, indexa en memoria stop_times.txt 
    para búsquedas instantáneas de conexiones.

    `ruta` puede ser una carpeta descomprimida o un .zip; ambas rutas de
    lectura quedan soportadas.
    """
    print("⏳ Cargando tablas base del GTFS...", file=sys.stderr)
    if os.path.isdir(ruta):
        stops = leer_tabla_gtfs_carpeta(ruta, "stops.txt")
        shapes = leer_tabla_gtfs_carpeta(ruta, "shapes.txt")
        trips = leer_tabla_gtfs_carpeta(ruta, "trips.txt")
        routes = leer_tabla_gtfs_carpeta(ruta, "routes.txt")
        agency = leer_tabla_gtfs_carpeta_opcional(ruta, "agency.txt")
        
        stop_times = None
        if cargar_horarios:
            ruta_st = _ruta_stop_times(ruta)
            print("⏳ Cargando e indexando 'stop_times.txt' en memoria (esto toma unos segundos la primera vez, pero acelera las consultas)...", file=sys.stderr)
            stop_times = pd.read_csv(ruta_st, dtype={"trip_id": str, "stop_id": str})
    else:
        with zipfile.ZipFile(ruta) as z:
            stops = leer_tabla_gtfs(z, "stops.txt")
            shapes = leer_tabla_gtfs(z, "shapes.txt")
            trips = leer_tabla_gtfs(z, "trips.txt")
            routes = leer_tabla_gtfs(z, "routes.txt")
            try:
                agency = leer_tabla_gtfs(z, "agency.txt")
            except FileNotFoundError:
                agency = None
            
            stop_times = None
            if cargar_horarios:
                print("⏳ Cargando e indexando 'stop_times.txt' desde ZIP...", file=sys.stderr)
                stop_times = leer_tabla_gtfs(z, "stop_times.txt")

    if agency is not None and "agency_id" in routes.columns:
        agency = agency.copy()
        agency["agency_id"] = agency["agency_id"].astype(str)
        routes = routes.copy()
        routes["agency_id"] = routes["agency_id"].astype(str)
        routes = routes.merge(agency[["agency_id", "agency_name"]], on="agency_id", how="left")
    else:
        routes = routes.copy()
        routes["agency_name"] = "Desconocida"

    return {"stops": stops, "shapes": shapes, "trips": trips, "routes": routes, "agency": agency, "stop_times": stop_times}


def construir_lineas_por_ruta(gtfs: dict, transformer: Transformer):
    """Construye una LineString (proyectada al CRS métrico) por cada shape_id del GTFS."""
    shapes = gtfs["shapes"].sort_values(["shape_id", "shape_pt_sequence"])
    trips = gtfs["trips"].dropna(subset=["shape_id"]).drop_duplicates("shape_id")
    shape_a_route = dict(zip(trips["shape_id"], trips["route_id"]))
    routes = gtfs["routes"].set_index("route_id")

    def nombre_ruta(route_id):
        if route_id in routes.index:
            row = routes.loc[route_id]
            corto = row.get("route_short_name", "")
            largo = row.get("route_long_name", "")
            return f"{corto} - {largo}".strip(" -") or str(route_id)
        return str(route_id)

    lineas, etiquetas = [], []
    for shape_id, grupo in shapes.groupby("shape_id"):
        if len(grupo) < 2:
            continue
        xs, ys = transformer.transform(grupo["shape_pt_lon"].values, grupo["shape_pt_lat"].values)
        lineas.append(LineString(zip(xs, ys)))
        route_id = shape_a_route.get(shape_id)
        etiquetas.append({
            "shape_id": shape_id,
            "route_id": route_id,
            "route_name": nombre_ruta(route_id),
        })
    return lineas, etiquetas


def calcular_distancia_a_rutas(estudiantes: pd.DataFrame, lineas, etiquetas, transformer):
    """Para cada estudiante, encuentra la ruta (línea) de transporte más cercana y su distancia en km."""
    if not lineas:
        estudiantes["ruta_mas_cercana"] = None
        estudiantes["dist_ruta_km"] = np.nan
        return estudiantes

    arbol = STRtree(lineas)
    xs, ys = transformer.transform(estudiantes["lon"].values, estudiantes["lat"].values)

    rutas_cercanas, distancias = [], []
    for x, y in zip(xs, ys):
        punto = Point(x, y)
        i_mas_cercano = arbol.nearest(punto)
        dist_m = punto.distance(lineas[i_mas_cercano])
        rutas_cercanas.append(etiquetas[i_mas_cercano]["route_name"])
        distancias.append(dist_m / 1000.0)

    estudiantes = estudiantes.copy()
    estudiantes["ruta_mas_cercana"] = rutas_cercanas
    estudiantes["dist_ruta_km"] = np.round(distancias, 3)
    return estudiantes


def calcular_distancia_a_paradas(estudiantes: pd.DataFrame, stops: pd.DataFrame, transformer):
    """Para cada estudiante, encuentra el paradero más cercano y su distancia en km (cKDTree)."""
    sx, sy = transformer.transform(stops["stop_lon"].values, stops["stop_lat"].values)
    arbol = cKDTree(np.column_stack([sx, sy]))
    ex, ey = transformer.transform(estudiantes["lon"].values, estudiantes["lat"].values)
    dist_m, idx = arbol.query(np.column_stack([ex, ey]))

    estudiantes = estudiantes.copy()
    estudiantes["parada_mas_cercana"] = stops["stop_name"].values[idx]
    estudiantes["dist_parada_km"] = np.round(dist_m / 1000.0, 3)
    return estudiantes


def _tiempo_a_segundos(t: str) -> int:
    h, m, s = map(int, t.split(":"))
    return h * 3600 + m * 60 + s


def _ruta_stop_times(carpeta_o_zip: str) -> str:
    if os.path.isdir(carpeta_o_zip):
        exacto = os.path.join(carpeta_o_zip, "stop_times.txt")
        if os.path.isfile(exacto):
            return exacto
        candidatos = [f for f in os.listdir(carpeta_o_zip) if f.endswith("stop_times.txt")]
        if candidatos:
            return os.path.join(carpeta_o_zip, candidatos[0])
    raise FileNotFoundError("No se encontró stop_times.txt.")


def _distancia_km(transformer, lat1, lon1, lat2, lon2):
    x1, y1 = transformer.transform(lon1, lat1)
    x2, y2 = transformer.transform(lon2, lat2)
    return ((x1 - x2) ** 2 + (y1 - y2) ** 2) ** 0.5 / 1000.0


def paradas_cercanas(lat, lon, stops, transformer, radio_km=0.6, max_n=5):
    """Devuelve los paraderos dentro de `radio_km` de un punto, ordenados por cercanía."""
    sx, sy = transformer.transform(stops["stop_lon"].values, stops["stop_lat"].values)
    x, y = transformer.transform(lon, lat)
    d = np.sqrt((sx - x) ** 2 + (sy - y) ** 2)
    cand = stops.assign(dist_km=d / 1000.0)
    return cand[cand["dist_km"] <= radio_km].sort_values("dist_km").head(max_n)


# ---------------------------------------------------------------------------
# BÚSQUEDA OPTIMIZADA: AHORA CONSULTA DESDE MEMORIA RAM 🚀
# ---------------------------------------------------------------------------
def buscar_mejor_conexion(
    gtfs, transformer,
    lat_origen, lon_origen, lat_destino, lon_destino,
    hora_min="06:00:00", hora_max="09:00:00", radio_km=0.6,
):
    """
    Busca, para un origen y un destino dados, la mejor conexión de transporte
    público (menor tiempo total) dentro de una ventana horaria [hora_min, hora_max].

    Evalúa primero viajes directos y, si no encuentra ninguno, viajes con un
    transbordo. Consulta siempre sobre las tablas GTFS ya cargadas en RAM.
    """
    stops, trips, routes = gtfs["stops"], gtfs["trips"], gtfs["routes"]
    stop_times = gtfs.get("stop_times")
    
    if stop_times is None:
        raise ValueError("No has cargado los horarios en memoria. Asegúrate de usar cargar_gtfs(..., cargar_horarios=True)")

    info_ruta = routes.set_index("route_id")[["route_short_name", "route_long_name", "agency_name"]]

    def _info_tramo(trip_id):
        route_id = trips.loc[trips["trip_id"] == trip_id, "route_id"]
        if route_id.empty or route_id.iloc[0] not in info_ruta.index:
            return {"ruta": None, "nombre_ruta": None, "agencia": None}
        r = info_ruta.loc[route_id.iloc[0]]
        return {"ruta": r["route_short_name"], "nombre_ruta": r["route_long_name"], "agencia": r["agency_name"]}

    p_origen = paradas_cercanas(lat_origen, lon_origen, stops, transformer, radio_km)
    p_destino = paradas_cercanas(lat_destino, lon_destino, stops, transformer, radio_km)
    if p_origen.empty or p_destino.empty:
        return None  # No hay paradas cercanas en origen o destino

    origen_ids = set(p_origen["stop_id"])
    destino_ids = set(p_destino["stop_id"])
    todos_ids = origen_ids | destino_ids
    
    dist_a_pie_origen = p_origen.set_index("stop_id")["dist_km"].to_dict()
    dist_a_pie_destino = p_destino.set_index("stop_id")["dist_km"].to_dict()
    nombre_parada = stops.set_index("stop_id")["stop_name"].to_dict()
    coords_parada = stops.set_index("stop_id")[["stop_lat", "stop_lon"]]

    # Filtrado instantáneo en memoria RAM (No más lecturas del disco duro 💽❌)
    relevante = stop_times[stop_times["stop_id"].isin(todos_ids)]
    if relevante.empty:
        return None

    salida_min_s, salida_max_s = _tiempo_a_segundos(hora_min), _tiempo_a_segundos(hora_max)

    # --- Intento 1: viaje directo ---
    dep_origen = relevante[relevante["stop_id"].isin(origen_ids)][["trip_id", "stop_id", "stop_sequence", "departure_time"]]
    dep_origen = dep_origen.rename(columns={"stop_id": "parada_origen", "stop_sequence": "seq_o", "departure_time": "salida_origen"})
    
    arr_destino = relevante[relevante["stop_id"].isin(destino_ids)][["trip_id", "stop_id", "stop_sequence", "arrival_time"]]
    arr_destino = arr_destino.rename(columns={"stop_id": "parada_destino", "stop_sequence": "seq_d", "arrival_time": "llegada_destino"})
    
    directos = dep_origen.merge(arr_destino, on="trip_id")
    directos = directos[directos["seq_o"] < directos["seq_d"]].copy()
    
    if not directos.empty:
        directos["salida_s"] = directos["salida_origen"].apply(_tiempo_a_segundos)
        directos = directos[(directos["salida_s"] >= salida_min_s) & (directos["salida_s"] <= salida_max_s)]
        if not directos.empty:
            directos["llegada_s"] = directos["llegada_destino"].apply(_tiempo_a_segundos)
            directos["tiempo_total_min"] = (directos["llegada_s"] - directos["salida_s"]) / 60
            mejor = directos.sort_values("tiempo_total_min").iloc[0]
            po, pd_ = mejor["parada_origen"], mejor["parada_destino"]
            co, cd = coords_parada.loc[po], coords_parada.loc[pd_]
            info = _info_tramo(mejor["trip_id"])
            return {
                "tipo": "directo",
                "parada_partida": nombre_parada.get(po, po),
                "lat_partida": co["stop_lat"], "lon_partida": co["stop_lon"],
                "dist_casa_a_parada_partida_km": round(dist_a_pie_origen.get(po, float("nan")), 3),
                "salida_origen": mejor["salida_origen"],
                "ruta": info["ruta"], "nombre_ruta": info["nombre_ruta"], "agencia": info["agencia"],
                "trip_id": mejor["trip_id"],
                "parada_final": nombre_parada.get(pd_, pd_),
                "lat_final": cd["stop_lat"], "lon_final": cd["stop_lon"],
                "llegada_destino": mejor["llegada_destino"],
                "dist_parada_final_a_uni_km": round(dist_a_pie_destino.get(pd_, float("nan")), 3),
                "dist_en_bus_km": round(_distancia_km(transformer, co["stop_lat"], co["stop_lon"], cd["stop_lat"], cd["stop_lon"]), 2),
                "dist_total_estimada_km": round(dist_a_pie_origen.get(po, 0) + dist_a_pie_destino.get(pd_, 0) + _distancia_km(transformer, co["stop_lat"], co["stop_lon"], cd["stop_lat"], cd["stop_lon"]), 2),
                "tiempo_total_min": round(mejor["tiempo_total_min"], 1),
                "transbordos": 0,
            }

    # --- Intento 2: un transbordo (usando filtrado optimizado de trips) ---
    trips_interes = set(dep_origen["trip_id"]) | set(arr_destino["trip_id"])
    completo_filtrado = stop_times[stop_times["trip_id"].isin(trips_interes)]

    t1 = dep_origen.merge(completo_filtrado[["trip_id", "stop_id", "stop_sequence", "arrival_time"]], on="trip_id")
    t1 = t1[t1["stop_sequence"] > t1["seq_o"]].copy()
    t1["salida_s"] = t1["salida_origen"].apply(_tiempo_a_segundos)
    t1 = t1[(t1["salida_s"] >= salida_min_s) & (t1["salida_s"] <= salida_max_s)]
    t1["llegada_transfer_s"] = t1["arrival_time"].apply(_tiempo_a_segundos)
    t1 = t1.rename(columns={"stop_id": "parada_transfer", "arrival_time": "hora_llegada_transfer"})

    t2 = arr_destino.merge(completo_filtrado[["trip_id", "stop_id", "stop_sequence", "departure_time"]], on="trip_id")
    t2 = t2[t2["stop_sequence"] < t2["seq_d"]].copy()
    t2["llegada_s"] = t2["llegada_destino"].apply(_tiempo_a_segundos)
    t2["salida_transfer_s"] = t2["departure_time"].apply(_tiempo_a_segundos)
    t2 = t2.rename(columns={"stop_id": "parada_transfer", "departure_time": "hora_salida_transfer"})

    cruce = t1.merge(t2, on="parada_transfer", suffixes=("_1", "_2"))
    cruce = cruce[cruce["salida_transfer_s"] >= cruce["llegada_transfer_s"]]
    if cruce.empty:
        return None

    cruce["espera_min"] = (cruce["salida_transfer_s"] - cruce["llegada_transfer_s"]) / 60
    cruce["tiempo_total_min"] = (cruce["llegada_s"] - cruce["salida_s"]) / 60
    mejor = cruce.sort_values("tiempo_total_min").iloc[0]

    po, pt, pd_ = mejor["parada_origen"], mejor["parada_transfer"], mejor["parada_destino"]
    co, ct, cd = coords_parada.loc[po], coords_parada.loc[pt], coords_parada.loc[pd_]
    dist_tramo1 = _distancia_km(transformer, co["stop_lat"], co["stop_lon"], ct["stop_lat"], ct["stop_lon"])
    dist_tramo2 = _distancia_km(transformer, ct["stop_lat"], ct["stop_lon"], cd["stop_lat"], cd["stop_lon"])
    info1 = _info_tramo(mejor["trip_id_1"])
    info2 = _info_tramo(mejor["trip_id_2"])

    return {
        "tipo": "con_transbordo",
        "parada_partida": nombre_parada.get(po, po),
        "lat_partida": co["stop_lat"], "lon_partida": co["stop_lon"],
        "dist_casa_a_parada_partida_km": round(dist_a_pie_origen.get(po, 0), 3),
        "salida_origen": mejor["salida_origen"],
        "ruta_tramo1": info1["ruta"], "nombre_ruta_tramo1": info1["nombre_ruta"], "trip_id_1": mejor["trip_id_1"],
        "parada_transbordo": nombre_parada.get(pt, pt), "lat_transbordo": ct["stop_lat"], "lon_transbordo": ct["stop_lon"],
        "dist_tramo1_bus_km": round(dist_tramo1, 2), "llegada_transbordo": mejor["hora_llegada_transfer"],
        "espera_transbordo_min": round(mejor["espera_min"], 1), "salida_transbordo": mejor["hora_salida_transfer"],
        "ruta_tramo2": info2["ruta"], "nombre_ruta_tramo2": info2["nombre_ruta"], "trip_id_2": mejor["trip_id_2"],
        "parada_final": nombre_parada.get(pd_, pd_), "lat_final": cd["stop_lat"], "lon_final": cd["stop_lon"],
        "dist_tramo2_bus_km": round(dist_tramo2, 2), "llegada_destino": mejor["llegada_destino"],
        "dist_parada_final_a_uni_km": round(dist_a_pie_destino.get(pd_, 0), 3),
        "dist_total_estimada_km": round(dist_a_pie_origen.get(po, 0) + dist_tramo1 + dist_tramo2 + dist_a_pie_destino.get(pd_, 0), 2),
        "tiempo_total_min": round(mejor["tiempo_total_min"], 1),
        "transbordos": 1,
    }


In [2]:
# ---------------------------------------------------------------------------
# CONFIGURACIÓN DE RUTAS (RELATIVAS AL REPOSITORIO) 📁
# ---------------------------------------------------------------------------
# Este notebook vive en `src/`, dentro del repo `Data Jam/`. Las rutas de
# abajo se calculan de forma relativa para que el notebook corra igual sin
# importar en qué máquina o carpeta se clone el repositorio.
#
# Estructura esperada:
#   Data Jam/
#   ├── data/
#   │   └── raw/
#   │       ├── GTFS-2026-04-29/      <- carpeta del feed GTFS (SITP + TransMilenio)
#   │       └── estudiantes.xlsx      <- coordenadas de vivienda y de la IES por estudiante
#   ├── outputs/                      <- aquí se guarda el resultado (estudiantes_con_rutas.xlsx)
#   └── src/
#       └── rutas_distancias_tiempo.ipynb   <- este notebook

from pathlib import Path

# Carpeta donde vive este notebook (src/). Si el notebook se ejecuta en un
# entorno donde `__file__` no existe (Jupyter clásico), se usa cwd() como
# respaldo.
try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd().resolve()

# La raíz del repo es la carpeta padre de src/ (si el notebook se ejecuta
# desde dentro de src/); si por algún motivo se ejecuta desde la raíz del
# repo, BASE_DIR queda igual a NOTEBOOK_DIR.
BASE_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "src" else NOTEBOOK_DIR

DATA_DIR = BASE_DIR / "data" / "raw"
OUTPUTS_DIR = BASE_DIR / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

RUTA_GTFS = DATA_DIR / "GTFS-2026-04-29"
RUTA_ESTUDIANTES = DATA_DIR / "estudiantes.xlsx"
RUTA_SALIDA = OUTPUTS_DIR / "estudiantes_con_rutas_v2.xlsx"

print(f"📁 Repositorio detectado en: {BASE_DIR}")
print(f"📂 GTFS esperado en:         {RUTA_GTFS}")
print(f"📄 Estudiantes esperado en:  {RUTA_ESTUDIANTES}")
print(f"💾 Salida se guardará en:    {RUTA_SALIDA}")


📁 Repositorio detectado en: C:\Users\Max\Documents\Atenea\Data Jam
📂 GTFS esperado en:         C:\Users\Max\Documents\Atenea\Data Jam\data\raw\GTFS-2026-04-29
📄 Estudiantes esperado en:  C:\Users\Max\Documents\Atenea\Data Jam\data\raw\estudiantes.xlsx
💾 Salida se guardará en:    C:\Users\Max\Documents\Atenea\Data Jam\outputs\estudiantes_con_rutas_v2.xlsx


In [3]:
# ---------------------------------------------------------------------------
# EJECUCIÓN MASIVA EN BUCLE 🚀 (CON PROGRESO Y LINKS DE MAPS)
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    import time

    # 1. Cargar el GTFS completo e indexar stop_times en RAM (Una sola vez)
    gtfs = cargar_gtfs(str(RUTA_GTFS), cargar_horarios=True)
    transformer = Transformer.from_crs(CRS_GEO, CRS_METRICO, always_xy=True)

    # 2. Construir geometrías espaciales
    lineas, etiquetas = construir_lineas_por_ruta(gtfs, transformer)
    estudiantes = pd.read_excel(RUTA_ESTUDIANTES).head(2)

    print("\n⚡ Calculando distancias espaciales iniciales...", flush=True)
    estudiantes = calcular_distancia_a_rutas(estudiantes, lineas, etiquetas, transformer)
    estudiantes = calcular_distancia_a_paradas(estudiantes, gtfs["stops"], transformer)

    # 3. Bucle masivo para calcular la mejor conexión por horarios
    total_estudiantes = len(estudiantes)
    print(f"\n⏱️ Iniciando cálculo de rutas/horarios para {total_estudiantes} estudiantes...", flush=True)
    inicio_bucle = time.time()
    
    lista_resultados = []

    # Intentar importar tqdm para una barra de progreso visual, si no, usar contador clásico
    try:
        from tqdm import tqdm
        iterador_estudiantes = tqdm(estudiantes.iterrows(), total=total_estudiantes, desc="Procesando estudiantes")
        usando_tqdm = True
    except ImportError:
        iterador_estudiantes = estudiantes.iterrows()
        usando_tqdm = False

    for idx, fila in iterador_estudiantes:
        # Contador visual alternativo en consola si no está instalado tqdm
        if not usando_tqdm and idx % 50 == 0:
            print(f"⏳ Procesados: {idx}/{total_estudiantes} estudiantes... ({ (idx/total_estudiantes)*100:.1f}%)", flush=True)

        # Consultar la conexión usando las coordenadas de vivienda y universidad de cada fila
        res_conexion = buscar_mejor_conexion(
            gtfs, transformer,
            lat_origen=fila["lat"], lon_origen=fila["lon"],
            lat_destino=fila["uni_lat"], lon_destino=fila["uni_lon"],
            hora_min="04:00:00", hora_max="10:00:00", radio_km=3,
        )
        
        # Si no encuentra ruta, creamos un dict vacío con la estructura básica
        if res_conexion is None:
            res_conexion = {
                "tipo": "No encontrado", 
                "tiempo_total_min": np.nan, 
                "transbordos": np.nan,
                "ruta_info": "Sin conexión en la ventana horaria",
                "link_maps": "Sin ruta"
            }
        else:
            # 1. Formatear la información de la ruta y construir URLs oficiales de Google Maps
            lat_o, lon_o = fila['lat'], fila['lon']
            lat_d, lon_d = fila['uni_lat'], fila['uni_lon']
            
            if res_conexion["tipo"] == "directo":
                res_conexion["ruta_info"] = f"Directo: {res_conexion['ruta']}"
                # Estructura oficial URL Google Maps: Origen -> Destino
                link = f"https://www.google.com/maps/dir/?api=1&origin={lat_o},{lon_o}&destination={lat_d},{lon_d}&travelmode=transit"
            
            else:
                res_conexion["ruta_info"] = f"Transbordo: {res_conexion['ruta_tramo1']} -> {res_conexion['ruta_tramo2']}"
                
                # Extraer coordenadas de la parada intermedia calculada en la RAM
                lat_t = res_conexion["lat_transbordo"]
                lon_t = res_conexion["lon_transbordo"]
                
                # Estructura oficial URL Google Maps con parada intermedia (waypoint)
                link = f"https://www.google.com/maps/dir/?api=1&origin={lat_o},{lon_o}&destination={lat_d},{lon_d}&waypoints={lat_t},{lon_t}&travelmode=transit"
            
            # Asignar el link correspondiente al diccionario de resultados
            res_conexion["link_maps"] = link

        lista_resultados.append(res_conexion)

    # 4. Convertir los diccionarios en un DataFrame y unirlos al original
    df_conexiones = pd.DataFrame(lista_resultados)
    
    # Prefijamos las nuevas columnas para mantener el orden y claridad
    df_conexiones = df_conexiones.add_prefix("conn_")
    
    # Combinamos horizontalmente con el DataFrame original de estudiantes
    estudiantes_final = pd.concat([estudiantes, df_conexiones], axis=1)
    
    fin_bucle = time.time()
    print(f"\n✅ ¡Procesamiento completo en {fin_bucle - inicio_bucle:.2f} segundos! 🏁", flush=True)

    # 5. Guardar el reporte unificado en outputs/ (ruta relativa al repo)
    estudiantes_final.to_excel(RUTA_SALIDA, index=False)
    print(f"💾 Archivo guardado exitosamente en: {RUTA_SALIDA}")

    # Mostrar vista previa del resultado (incluyendo el link generado)
    # Ajustado para usar las columnas reales que existan en tu dataframe
    columnas_vista = ["lat", "lon", "conn_tipo", "conn_tiempo_total_min", "conn_ruta_info", "conn_link_maps"]
    columnas_existentes = [col for col in columnas_vista if col in estudiantes_final.columns]
    
    print("\n👀 Vista previa de los resultados:")
    print(estudiantes_final[columnas_existentes].head(10).to_string(index=False))


⏳ Cargando tablas base del GTFS...
⏳ Cargando e indexando 'stop_times.txt' en memoria (esto toma unos segundos la primera vez, pero acelera las consultas)...



⚡ Calculando distancias espaciales iniciales...

⏱️ Iniciando cálculo de rutas/horarios para 2 estudiantes...


Procesando estudiantes: 100%|██████████| 2/2 [00:01<00:00,  1.72it/s]


✅ ¡Procesamiento completo en 1.19 segundos! 🏁
💾 Archivo guardado exitosamente en: C:\Users\Max\Documents\Atenea\Data Jam\outputs\estudiantes_con_rutas_v2.xlsx

👀 Vista previa de los resultados:
     lat        lon      conn_tipo  conn_tiempo_total_min          conn_ruta_info                                                                                                                                                                      conn_link_maps
4.670288 -74.080814        directo                   10.6          Directo: FA427                                                 https://www.google.com/maps/dir/?api=1&origin=4.67028789400001,-74.080814051&destination=4.66076807,-74.05981642&travelmode=transit
4.597230 -74.147068 con_transbordo                   85.8 Transbordo: 97 -> HA680 https://www.google.com/maps/dir/?api=1&origin=4.59722965600002,-74.147068184&destination=4.60043585,-74.07310909&waypoints=4.5886928134454426,-74.15066952786681&travelmode=transit
